# What is Jev? TypeSafe's System One model, paired with Claude, on Strands

The notebook from the video, rebuilt on the [Strands Agents](https://strandsagents.com) SDK, with Claude on Amazon Bedrock. Two ways to run the same agent.

The agent answers support questions from eight read-only lookups (`tools.py`). Each lookup has a simulated 1.5 s round trip.

1. **Claude alone**: a Strands `Agent` with the eight lookups as tools. Claude Sonnet 5 picks each lookup, waits for it, and repeats. Three of every four Claude calls only decide what to look up next.
2. **Jev fetches, Claude writes**: a Strands `Agent` with no tools, plus a hook. Before the invocation, the hook asks Jev to score every possible lookup in one call, runs the likely ones, and rewrites the prompt to the task plus the facts. Claude answers in a single call with no tools and no schemas. After the invocation, if Claude replied `MISSING:`, the same hook falls back to the Claude-alone agent.

Nothing in the tools or the agent loop changes between the two arms. Everything the harness does, the counting, the step limit, Jev's prefetch and the fallback, lives in Strands hooks:

| hook event | arm 1 | arm 2 |
|---|---|---|
| `BeforeInvocationEvent` | start the clock | start the clock; Jev picks lookups, harness runs them, prompt becomes task + facts |
| `BeforeModelCallEvent` | count Claude calls, stop after 8 | same |
| `AfterToolCallEvent` | count lookups | (no tools) |
| `AfterInvocationEvent` | tokens, seconds, answer | on `MISSING:`, run arm 1 and keep its answer |

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from strands import Agent
from strands.models import BedrockModel
from strands.hooks import (
    AfterInvocationEvent, AfterToolCallEvent,
    BeforeInvocationEvent, BeforeModelCallEvent, HookProvider, HookRegistry,
)
from typesafe_sdk import TypeSafeClient

from agent_jev_helper import JevStats, candidate_calls, question
from tools import STRANDS_TOOLS, call_tool

load_dotenv(".env")   # before reading BEDROCK_MODEL_ID and AWS_REGION

MODEL = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-sonnet-5")
REGION = os.getenv("AWS_REGION", "us-east-1")
USD_PER_MILLION = {"claude-sonnet-5": (2.0, 10.0), "claude-haiku-4-5": (1.0, 5.0)}   # on-demand, per million tokens in / out
MAX_TOKENS = 4096
MAX_STEPS = 8
REPEATS = 3


def make_model(model: str = MODEL) -> BedrockModel:
    """Claude on Amazon Bedrock, with the AWS credentials of your environment or AWS_PROFILE."""
    return BedrockModel(model_id=model, region_name=REGION, max_tokens=MAX_TOKENS)


tasks = json.load(open("tasks.json"))
task = tasks[0]["task"]
print(MODEL)
task

## The run's numbers, collected by hooks

`RunResult` is a Strands `HookProvider`. Hand it to an agent and it records the run: Claude calls, lookups, tokens, seconds and the answer. It also enforces the step limit by cancelling the ninth model call.

In [ ]:
@dataclass
class RunResult(HookProvider):
    model: str = MODEL
    seconds: float = 0.0
    claude_calls: int = 0
    tool_calls: int = 0
    input_tokens: int = 0
    output_tokens: int = 0
    answer: str = ""
    started: float = field(default=0.0, repr=False)

    def register_hooks(self, registry: HookRegistry, **_: Any) -> None:
        registry.add_callback(BeforeInvocationEvent, self.start)
        registry.add_callback(BeforeModelCallEvent, self.count_model_call)
        registry.add_callback(AfterToolCallEvent, self.count_tool_call)
        registry.add_callback(AfterInvocationEvent, self.finish)

    def start(self, event: BeforeInvocationEvent) -> None:
        self.started = time.perf_counter()

    def count_model_call(self, event: BeforeModelCallEvent) -> None:
        if self.claude_calls >= MAX_STEPS:
            event.cancel = f"Stopped after {MAX_STEPS} Claude calls."
        else:
            self.claude_calls += 1

    def count_tool_call(self, event: AfterToolCallEvent) -> None:
        self.tool_calls += 1

    def finish(self, event: AfterInvocationEvent) -> None:
        if event.result is None:   # the invocation raised; let its own error surface
            return
        usage = event.result.metrics.accumulated_usage
        self.input_tokens += usage["inputTokens"]
        self.output_tokens += usage["outputTokens"]
        self.answer = self.answer or str(event.result).strip()   # a fallback may already have set it
        self.seconds = time.perf_counter() - self.started

    @property
    def claude_usd(self) -> float:
        usd_in, usd_out = next(price for name, price in USD_PER_MILLION.items() if name in self.model)
        return (self.input_tokens * usd_in + self.output_tokens * usd_out) / 1_000_000

## Arm 1: Claude alone

A plain Strands agent: the eight lookups as tools, and the hook above. Strands runs the lookups of one turn in parallel.

In [ ]:
SYSTEM = (
    "You are a support operations agent. Gather facts with the lookup tools, "
    "calling independent lookups in parallel in one turn. "
    "Then answer in three sentences or fewer."
)


def run_agent(task: str, model: str = MODEL) -> RunResult:
    result = RunResult(model)
    agent = Agent(model=make_model(model), tools=STRANDS_TOOLS, system_prompt=SYSTEM, hooks=[result], callback_handler=None)
    agent(task)
    return result

## Arm 2: Jev fetches, Claude writes

`gather_facts` is the harness: round by round, it asks Jev one yes/no question per candidate lookup (`agent_jev_helper.py`), runs every lookup scoring 0.7 or above in parallel, and repeats while the results expose new ids.

`JevFetches` puts it in front of Claude as a hook. Claude's agent has no tools at all, so its one call carries no schemas and no history.

In [ ]:
FETCH_AT = 0.7
MAX_ROUNDS = 4
MISSING = "MISSING:"
WRITER_SYSTEM = (
    "You are a support operations agent. Answer the task from the facts given, in three sentences or fewer. "
    f"If a fact you need is not in the list, reply with only '{MISSING} <what you need>'."
)


def gather_facts(task: str, jev: TypeSafeClient, stats: JevStats) -> list[str]:
    """Round by round, run every lookup Jev rates likely, until nothing is."""
    history: list[str] = []
    called: set[tuple[str, str]] = set()
    with ThreadPoolExecutor() as pool:
        for _ in range(MAX_ROUNDS):
            state: dict[str, Any] = {"task": task, "lookups_so_far": history}
            calls = candidate_calls(state, called)
            if not calls:
                break
            response = jev.system_one(state=state, questions={f"q{i}": question(c) for i, c in enumerate(calls)})
            stats.add_usage(response.usage.input_tokens)
            chosen = [call for i, call in enumerate(calls) if response.answers[f"q{i}"].noul >= FETCH_AT]
            if not chosen:
                break
            results = pool.map(call_tool, *zip(*chosen))
            called.update(chosen)
            history += [f"{name}({value}) -> {out}" for (name, value), out in zip(chosen, results)]
    return history

In [ ]:
class JevFetches(HookProvider):
    """Before the invocation: Jev picks the lookups, the harness runs them, the prompt becomes the task plus the facts.
    After it: on a MISSING reply, run the Claude-alone agent, so the answer is never written from missing facts."""

    def __init__(self, run: RunResult, jev: TypeSafeClient | None = None) -> None:
        self.run, self.jev, self.stats = run, jev or TypeSafeClient(), JevStats()
        self.task, self.fell_back = "", False

    def register_hooks(self, registry: HookRegistry, **_: Any) -> None:
        registry.add_callback(BeforeInvocationEvent, self.fetch)
        registry.add_callback(AfterInvocationEvent, self.fall_back_if_missing)

    def fetch(self, event: BeforeInvocationEvent) -> None:
        self.task = event.messages[-1]["content"][0]["text"]
        facts = gather_facts(self.task, self.jev, self.stats)
        self.run.tool_calls += len(facts)
        prompt = f"Task: {self.task}\n\nFacts:\n" + "\n".join(f"- {fact}" for fact in facts)
        event.messages = [{"role": "user", "content": [{"text": prompt}]}]

    def fall_back_if_missing(self, event: AfterInvocationEvent) -> None:
        self.fell_back = str(event.result).strip().startswith(MISSING)
        if self.fell_back:
            fallback = run_agent(self.task, model=self.run.model)
            self.run.answer = fallback.answer
            self.run.tool_calls += fallback.tool_calls
            self.run.claude_calls += fallback.claude_calls
            self.run.input_tokens += fallback.input_tokens
            self.run.output_tokens += fallback.output_tokens


def run_agent_jev_first(task: str, model: str = MODEL) -> tuple[RunResult, JevStats, bool]:
    """Returns the run, Jev's stats, and whether Claude had to fall back to the tool-using agent."""
    result = RunResult(model)
    jev = JevFetches(result)
    # Hook order matters: after-invocation callbacks run in reverse, so the fallback lands before RunResult stops the clock.
    agent = Agent(model=make_model(model), system_prompt=WRITER_SYSTEM, hooks=[result, jev], callback_handler=None)
    agent(task)
    return result, jev.stats, jev.fell_back

## One task, Claude alone

In [ ]:
base = run_agent(task)
print(f"{base.seconds:.1f}s   claude calls={base.claude_calls}   lookups={base.tool_calls}   input tokens={base.input_tokens}   ${base.claude_usd:.4f}\n")
print(base.answer)

## The same task, Jev fetches and Claude writes

In [ ]:
first, s, fell_back = run_agent_jev_first(task)
print(f"{first.seconds:.1f}s   claude calls={first.claude_calls}   lookups={first.tool_calls}   input tokens={first.input_tokens}   ${first.claude_usd:.4f} + Jev ${s.jev_usd:.5f}   fell back={fell_back}\n")
print(first.answer)

## All tasks, both ways, repeated

In [ ]:
rows = []
for rep in range(REPEATS):
    for item in tasks:
        base = run_agent(item["task"])
        first, s, fell_back = run_agent_jev_first(item["task"])
        for arm, run, jev_usd, fb in [("Claude alone", base, 0.0, False), ("Jev fetches first", first, s.jev_usd, fell_back)]:
            rows.append({
                "arm": arm, "task": item["id"], "rep": rep, "seconds": run.seconds,
                "claude_calls": run.claude_calls, "input_tokens": run.input_tokens, "lookups": run.tool_calls,
                "claude_usd": run.claude_usd, "jev_usd": jev_usd, "usd": run.claude_usd + jev_usd, "fell_back": fb,
            })
df = pd.DataFrame(rows)
df.to_csv("runs.csv", index=False)
len(df)

### Median seconds per task

In [ ]:
ARMS = ["Claude alone", "Jev fetches first"]
seconds = df.pivot_table(index="task", columns="arm", values="seconds", aggfunc="median")[ARMS]
seconds["speedup"] = seconds["Claude alone"] / seconds["Jev fetches first"]
seconds.round(2)

In [ ]:
ax = seconds[ARMS].plot.bar(figsize=(10, 4), color=["#9a9a9a", "#c4622d"], rot=30)
ax.set_ylabel("median seconds per task")
plt.tight_layout()

### Claude calls, tokens and cost per task (medians)

In [ ]:
cost = df.pivot_table(index="arm", values=["seconds", "claude_calls", "input_tokens", "lookups", "claude_usd", "jev_usd", "usd"], aggfunc="median").loc[ARMS]
cost[["seconds", "claude_calls", "input_tokens", "lookups", "claude_usd", "jev_usd", "usd"]].round({"seconds": 2, "claude_usd": 5, "jev_usd": 6, "usd": 5})

In [ ]:
ax = cost[["claude_usd", "jev_usd"]].plot.bar(stacked=True, figsize=(6, 4), color=["#9a9a9a", "#c4622d"], rot=0)
ax.set_ylabel("median USD per task")
ax.legend(["Claude", "Jev"])
plt.tight_layout()

In [ ]:
total = df.groupby("arm")[["seconds", "usd"]].sum().loc[ARMS] / REPEATS
fallbacks = int(df[df.arm == "Jev fetches first"].fell_back.sum())
print(f"Time   {total.seconds['Claude alone']:.1f}s -> {total.seconds['Jev fetches first']:.1f}s   ({total.seconds['Claude alone'] / total.seconds['Jev fetches first']:.2f}x faster)")
print(f"Cost   ${total.usd['Claude alone']:.4f} -> ${total.usd['Jev fetches first']:.4f}   ({total.usd['Claude alone'] / total.usd['Jev fetches first']:.1f}x cheaper)")
print(f"Fell back to the tool-using agent in {fallbacks} of {len(tasks) * REPEATS} runs")

## Read this before trusting the numbers

- Medians over 3 runs. Claude takes different paths from run to run, and one slow response can flip a task.
- Tool latency is simulated at 1.5 s. Jev-first removes Claude round trips, so its gain grows as tools get faster.
- Only read-only lookups are fetched this way. Never let a predictor run a write.
- Jev reads the question literally. Asking "is X the very next lookup" halved its accuracy, because parallel lookups split the probability. Asking "is X one of the lookups to run now" fixed it.
- Claude cannot ask for more. The MISSING fallback catches under-fetching. It cannot catch a wrong answer written from incomplete facts.
- This runs Claude on Bedrock through Strands, not the Anthropic API directly, so seconds and tokens will not match the original `runs.csv` exactly. Costs use on-demand list prices; check Bedrock pricing for your region.